# 02 · Gene set overlap tests

**Question:** Does macrocephaly share more genes with microcephaly, autism and seizures than we would expect by chance?

**Why a test is needed:** raw overlap counts mislead. A large gene set (like seizure, ~2,100 genes) will overlap with almost anything just because of its size. The **hypergeometric test** asks: given the sizes of both sets and the total number of genes, how surprising is this overlap?

**Input:** `results/gene_sets/` from notebook 01  
**Outputs:** `results/overlap_tests.csv`, `results/pairwise_jaccard.csv`, `results/shared_genes.csv`, two figures in `figures/`

## 1. Setup and load the gene sets
If you run this in a new Colab session, first upload `results_01.zip` from notebook 01 (left sidebar → Files → upload), then run the unzip line.

In [ ]:
import os, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom

# Uncomment if starting a fresh session:
# !unzip -q -o results_01.zip

os.makedirs("figures", exist_ok=True)

def read_set(path):
    with open(path) as f:
        return {line.strip() for line in f if line.strip()}

LABELS = ["macrocephaly", "microcephaly", "autism", "seizure"]
sets = {lab: read_set(f"results/gene_sets/{lab}.txt") for lab in LABELS}
background = read_set("results/gene_sets/background_genes.txt")

N = len(background)
print(f"Background: {N:,} genes")
for lab, s in sets.items():
    print(f"{lab:<13} {len(s):>5}")

## 2. The test, written out

For two gene sets A and B drawn from a background of N genes:

- **Expected overlap** by chance = |A| × |B| / N
- **Fold enrichment** = observed overlap / expected overlap
- **Hypergeometric p-value** = probability of seeing *at least* this many shared genes if B were a random draw of |B| genes from the background
- **Jaccard index** = shared / union (a size-independent measure of similarity, 0 to 1)

`hypergeom.sf(k - 1, N, |A|, |B|)` gives P(overlap ≥ k).

In [ ]:
def overlap_stats(A, B, N):
    k = len(A & B)
    expected = len(A) * len(B) / N
    return {
        "size_A": len(A),
        "size_B": len(B),
        "overlap": k,
        "expected": round(expected, 1),
        "fold_enrichment": round(k / expected, 2) if expected else np.nan,
        "jaccard": round(k / len(A | B), 3),
        "p_value": hypergeom.sf(k - 1, N, len(A), len(B)),
    }

## 3. Macrocephaly vs each comparison phenotype
Three tests are run, so the p-values are corrected for multiple testing (Benjamini–Hochberg).

In [ ]:
def bh_adjust(p):
    p = np.asarray(p, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(ranked, 1.0)
    return out

rows = []
for other in LABELS[1:]:
    r = overlap_stats(sets["macrocephaly"], sets[other], N)
    r["comparison"] = f"macrocephaly vs {other}"
    rows.append(r)

tests = pd.DataFrame(rows).set_index("comparison")
tests["p_adj_BH"] = bh_adjust(tests["p_value"])
tests.to_csv("results/overlap_tests.csv")

pd.options.display.float_format = "{:.3g}".format
tests

**How to read this:** look at `fold_enrichment` together with `p_adj_BH`.  
A fold enrichment of 2 means twice as many shared genes as chance would give. A tiny p-value with a fold enrichment near 1 means the overlap is statistically real but biologically weak.

## 4. Figure: enrichment of shared genes

In [ ]:
plot_df = tests.copy()
plot_df["label"] = [c.replace("macrocephaly vs ", "") for c in plot_df.index]

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.barh(plot_df["label"], plot_df["fold_enrichment"], color="#4C72B0")
ax.axvline(1, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Fold enrichment of shared genes (observed / expected)")
ax.set_title("Gene overlap with macrocephaly")

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height() / 2,
            f"{row['overlap']} genes, adj. p = {row['p_adj_BH']:.1e}",
            va="center", fontsize=8)

ax.set_xlim(0, plot_df["fold_enrichment"].max() * 1.6)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("figures/overlap_enrichment.png", dpi=200)
plt.show()

## 5. All pairs: Jaccard similarity
This shows how similar every pair of phenotypes is, not only pairs involving macrocephaly.

In [ ]:
jac = pd.DataFrame(index=LABELS, columns=LABELS, dtype=float)
for a, b in itertools.product(LABELS, LABELS):
    jac.loc[a, b] = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
jac.to_csv("results/pairwise_jaccard.csv")

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(jac.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS)), LABELS, rotation=30, ha="right")
ax.set_yticks(range(len(LABELS)), LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        v = jac.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v > 0.5 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="Jaccard index")
ax.set_title("Gene set similarity between phenotypes")
plt.tight_layout()
plt.savefig("figures/jaccard_heatmap.png", dpi=200)
plt.show()

## 6. Which genes are shared?
Genes linked to macrocephaly **and** to all three comparison phenotypes are good candidates for shared neurodevelopmental mechanisms. These feed into the network analysis in notebook 04.

In [ ]:
macro = sets["macrocephaly"]
shared = pd.DataFrame({
    "gene": sorted(macro),
    "microcephaly": [g in sets["microcephaly"] for g in sorted(macro)],
    "autism":       [g in sets["autism"] for g in sorted(macro)],
    "seizure":      [g in sets["seizure"] for g in sorted(macro)],
})
shared["n_other_phenotypes"] = shared[["microcephaly", "autism", "seizure"]].sum(axis=1)
shared = shared.sort_values(["n_other_phenotypes", "gene"], ascending=[False, True])
shared.to_csv("results/shared_genes.csv", index=False)

print("Macrocephaly genes by number of other phenotypes they are also linked to:")
print(shared["n_other_phenotypes"].value_counts().sort_index())

in_all = shared.loc[shared["n_other_phenotypes"] == 3, "gene"].tolist()
print(f"\nShared with all three ({len(in_all)}):")
print(", ".join(in_all))

## 7. A caveat to keep in mind
HPO annotations come from published case reports. Well-studied genes collect many phenotype annotations simply because more papers describe them. This **annotation bias** can inflate overlaps between any two phenotypes.

Using only HPO-annotated genes as the background (rather than all ~20,000 human genes) reduces this effect, but does not remove it. Overlaps here show shared *reported* genetics, which is a starting point for biology, not proof of a shared mechanism.

## 8. Download the results

In [ ]:
!zip -qr results_02.zip results/ figures/
from google.colab import files
files.download("results_02.zip")